# W4D2 — A CNN on MNIST — Guided

**Week 4 · Day 2 · CNNs & Model Fine-Tuning** · Lab

Yesterday you chose nine numbers and the edges appeared. Today you choose none of them: sixteen
filters are initialised at random and gradient descent decides what they become. At the end you
will draw them, and some of them will be edge detectors — the same shapes you wrote by hand,
arrived at by a machine that was only ever told to name digits.

The accuracy target (≥ 0.98) is deliberately easy to hit. It is not the graded content. The graded
content is the arithmetic — **432 against 393,216** — and the flatten dimension, because those are
the two things that transfer to Thursday and to week 6.

<div dir="rtl" align="right">

# الأسبوع ٤ · اليوم ٢ — شبكة التفافية على MNIST

**الأسبوع الرابع · اليوم الثاني · الشبكات الالتفافية وضبط النماذج** · معمل

بالأمس اخترتَ تسعة أعداد فظهرت الحواف. واليوم لا تختار منها شيئًا: تُهيَّأ ستة عشر مرشِّحًا عشوائيًا
ويقرّر النزول الاشتقاقي ما تصير إليه. وفي النهاية سترسمها، وسيكون بعضها كواشف حواف — الأشكال نفسها
التي كتبتها بيدك، بلغتها آلة لم يُقَل لها إلا أن تسمّي الأرقام.

وهدف الدقة (٠٫٩٨ فأعلى) سهل البلوغ عمدًا، وليس هو المحتوى المُقيَّم. المُقيَّم هو الحساب — **٤٣٢ مقابل
٣٩٣٬٢١٦** — وبُعد التسطيح، فهذان ما ينتقلان إلى يوم الخميس وإلى الأسبوع السادس.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

By the end of this lab you can:

- Max-pool and average-pool a grid by hand and say which one you would carry forward, and why.
- Assemble a CNN as an `nn.Module` whose flatten dimension is **derived**, so the same model accepts
  a 28×28 and a 32×32 image without a line changing.
- Count the parameters of any conv layer, and say where a small CNN's parameters actually sit.
- Train it, read the two curves with W3D5's vocabulary, and say which of the four shapes you got.
- Find the two classes a model confuses and look at the examples before blaming the model.
- Draw the first layer's learned filters and recognise what they became.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تُجري التجميع الأقصى والمتوسّط على شبكة بيدك، وتقول أيّهما تحمل معك ولماذا.
- أن تركّب شبكة التفافية كـ`nn.Module` يكون بُعد التسطيح فيها **مُشتقًّا**، فيقبل النموذج نفسه صورة
  ٢٨×٢٨ وصورة ٣٢×٣٢ دون تغيير سطر.
- أن تعدّ معاملات أي طبقة التفافية، وتقول أين تجلس معاملات الشبكة الصغيرة فعلًا.
- أن تُدرّبها وتقرأ المنحنيين بمفردات الأسبوع الثالث اليوم الخامس، وتقول أي الأشكال الأربعة حصل عندك.
- أن تجد الفئتين اللتين يخلط بينهما النموذج وتنظر إلى الأمثلة قبل أن تلوم النموذج.
- أن ترسم مرشِّحات الطبقة الأولى المتعلَّمة وتتعرّف على ما صارت إليه.

</div>

## About the data

**Dataset:** `mnist` — 70,000 handwritten digits, 28×28, greyscale, 60,000 train / 10,000 test.
Downloaded by `torchvision` on first use (~11 MB) and cached; the stretch section adds
`fashion_mnist` (~30 MB). Both are `loader: torchvision` in the registry, so **not** `get_dataset`.

One row is one digit, written by a US Census Bureau employee or an American high-school student in
the early 1990s. The target is the digit, 0–9, and the classes are near-balanced.

**The known problem with it:** MNIST is unrealistically easy. Centred, size-normalised, one object
per image, no background, no colour, no occlusion. A logistic regression on raw pixels scores about
0.92. Today's CNN will pass 0.98 in two epochs. **A number quoted from MNIST tells you nothing
about a vision system** — which is exactly why the stretch section re-runs the identical
architecture on Fashion-MNIST and the accuracy falls off a cliff.

**Time:** training is the long pole. Five epochs on a laptop CPU took about **41 seconds** on the
machine this was written on, roughly 8 seconds an epoch. If your first epoch is still running after
three minutes, something is wrong — check that `num_workers=0` and that nothing else is competing
for the CPU.

<div dir="rtl" align="right">

## عن البيانات

**مجموعة البيانات:** `mnist` — سبعون ألف رقم مكتوب بخط اليد، ٢٨×٢٨، بتدرّج رمادي، ستون ألفًا للتدريب
وعشرة آلاف للاختبار. يُنزّلها `torchvision` عند أول استخدام (نحو ١١ ميغابايت) ويخزّنها، ويضيف القسم
الإضافي `fashion_mnist` (نحو ٣٠ ميغابايت). وكلتاهما `loader: torchvision` في السجلّ، فلا تُستدعيان
بـ`get_dataset`.

الصف الواحد رقم واحد، كتبه موظّف في مكتب الإحصاء الأمريكي أو طالب ثانوية أمريكي في أوائل التسعينيات.
والهدف هو الرقم من ٠ إلى ٩، والفئات شبه متوازنة.

**المشكلة المعروفة فيها:** أن MNIST سهلة على نحو غير واقعي. فالأرقام موسَّطة ومُوحَّدة الحجم، وفي كل
صورة كائن واحد، بلا خلفية ولا لون ولا حجب. والانحدار اللوجستي على البكسلات الخام يبلغ نحو ٠٫٩٢،
وشبكة اليوم ستتجاوز ٠٫٩٨ في حقبتين. **والرقم المقتبَس من MNIST لا يقول شيئًا عن نظام رؤية** — ولهذا
بالضبط يُعيد القسم الإضافي تشغيل البنية نفسها على Fashion-MNIST فتهوي الدقة.

**الزمن:** التدريب هو الحلقة الأبطأ. استغرقت خمس حقب على معالج حاسوب محمول نحو **٤١ ثانية** على
الجهاز الذي كُتب عليه هذا الدفتر، أي نحو ثماني ثوانٍ للحقبة. فإن كانت حقبتك الأولى ما تزال تعمل بعد
ثلاث دقائق فثمّة خطأ — تحقّق أن `num_workers=0` وأن لا شيء آخر يزاحمك على المعالج.

</div>

## Setup

<div dir="rtl" align="right">

## الإعداد

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import describe_dataset
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, check_shape, report

ensure("torchinfo", "matplotlib")        # ← only this lab's extra packages
seed_everything(42)                      # course-wide seed

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

EPOCHS = 5              # measured: reaches 0.9899 on the test set, ~8 s per epoch on a laptop CPU
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
MNIST_ROOT = ARTEFACT_DIR / "torchvision"

TRANSFORM = transforms.ToTensor()
train_set = datasets.MNIST(MNIST_ROOT, train=True, download=True, transform=TRANSFORM)
test_set = datasets.MNIST(MNIST_ROOT, train=False, download=True, transform=TRANSFORM)

print(describe_dataset("mnist"))
print(f"\ntrain {len(train_set)} | test {len(test_set)} | one image {train_set[0][0].shape}")
print(versions(), "| device:", device())

## Section 1 — Warm-up: pooling, and looking at the data  (≈25 min)

Everything here works. Two things: the pooling numbers from this morning, then nine digits on
screen, because you always look at the data before you model it.

Yesterday's output grid — three rows of `−3 0 +3` — pooled with a 2×2 window and stride 1. Max
pooling keeps the strongest response in each window; average pooling keeps the mean. The slide's
answers are `[[0, 3], [0, 3]]` and `[[-1.5, 1.5], [-1.5, 1.5]]`, and the third line is the one
people miss: **a pooling layer has no parameters at all.**

<div dir="rtl" align="right">

## القسم الأول — الإحماء: التجميع والنظر إلى البيانات (نحو ٢٥ دقيقة)

كل ما هنا يعمل. أمران: أرقام التجميع من هذا الصباح، ثم تسعة أرقام على الشاشة، لأنك تنظر إلى البيانات
دائمًا قبل أن تُنمذجها.

شبكة خرج الأمس — ثلاثة صفوف من `−٣ ٠ +٣` — مُجمَّعة بنافذة ٢×٢ وخطوة ١. فالتجميع الأقصى يُبقي أقوى
استجابة في كل نافذة، والتجميع المتوسّط يُبقي المتوسّط. وإجابتا الشريحة `[[0, 3], [0, 3]]` و
`[[-1.5, 1.5], [-1.5, 1.5]]`، والسطر الثالث هو ما يفوت الناس: **طبقة التجميع لا معاملات لها البتّة.**

</div>

In [ ]:
grid = torch.tensor([[-3.0, 0.0, 3.0]] * 3).view(1, 1, 3, 3)   # yesterday's output

print("yesterday's grid:\n", grid.view(3, 3).numpy())
print("\nmax-pooled  (2x2, stride 1):\n", nn.MaxPool2d(2, 1)(grid).view(2, 2).numpy())
print("\navg-pooled  (2x2, stride 1):\n", nn.AvgPool2d(2, 1)(grid).view(2, 2).numpy())
print("\nparameters in a pooling layer:",
      sum(p.numel() for p in nn.MaxPool2d(2, 1).parameters()))

**Change one thing:** swap `nn.MaxPool2d(2, 1)` for `nn.MaxPool2d(2)` — stride defaults to the
window size — and the 3×3 grid pools to a single 1×1. That is the *usual* setting, and it is why
two pooling layers turn a 28×28 image into 7×7.

<div dir="rtl" align="right">

**غيّر شيئًا واحدًا:** استبدل `nn.MaxPool2d(2)` بـ`nn.MaxPool2d(2, 1)` — فالخطوة تأخذ حجم النافذة
افتراضيًا — فتُجمَّع الشبكة ٣×٣ إلى ١×١ واحدة. وهذا هو الضبط **المعتاد**، وهو سبب تحوّل صورة ٢٨×٢٨
إلى ٧×٧ بعد طبقتَي تجميع.

</div>

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(6, 6))
for ax, index in zip(axes.ravel(), range(9)):
    image, label = train_set[index]
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(f"label: {label}")
    ax.axis("off")
plt.tight_layout()
plt.show()

print("pixel range:", float(train_set[0][0].min()), "to", float(train_set[0][0].max()))
print("class counts:", np.bincount(train_set.targets.numpy()))

## Section 2 — Core: six tasks  (≈60 min)

1. The CNN as an `nn.Module`, with a **derived** flatten dimension.
2. `torchinfo.summary`, and the 432-against-393,216 arithmetic.
3. Train it, recording four numbers an epoch.
4. Plot the curves and diagnose them in words.
5. Confusion matrix, and five pictures of the worst pair.
6. Draw the first layer's filters. Nobody programmed them.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. الشبكة الالتفافية كـ`nn.Module` ببُعد تسطيح **مُشتقّ**.
٢. `torchinfo.summary` وحساب ٤٣٢ مقابل ٣٩٣٬٢١٦.
٣. درّبها وسجّل أربعة أعداد في كل حقبة.
٤. ارسم المنحنيين وشخّصهما بالكلمات.
٥. مصفوفة الالتباس، وخمس صور لأسوأ زوج.
٦. ارسم مرشِّحات الطبقة الأولى. لم يبرمجها أحد.

</div>

### Task 2.1 — the model, and the flatten dimension you must not type

Two conv-ReLU-pool blocks, then a flatten, then two dense layers. The first conv layer uses 5×5
filters — big enough that you can *see* what they learn in task 2.6; the second uses 3×3.

**Do not hard-code the flatten dimension.** Every student writes `nn.Linear(1568, 64)` once, then
changes the image size and meets

```
RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x400 and 800x10)
```

— this morning's activity 1. There are two honest ways out. `nn.LazyLinear` infers the dimension
from the first batch it sees; `nn.AdaptiveAvgPool2d((7, 7))` goes further and *fixes* the grid the
head receives, whatever came in. Use the adaptive pool: it is what makes the sanity check's
"accepts 32×32 too" assertion pass, and it is exactly what `resnet18` does before its own classifier
— which is why Thursday's fine-tuning works on images that are not 224×224 either.

<div dir="rtl" align="right">

### المهمة ٢٫١ — النموذج، وبُعد التسطيح الذي يجب ألّا تكتبه

كتلتا التفاف-ReLU-تجميع، ثم تسطيح، ثم طبقتان كثيفتان. تستخدم الطبقة الالتفافية الأولى مرشِّحات ٥×٥ —
كبيرة بما يكفي لتـ**رى** ما تتعلّمه في المهمة ٢٫٦ — وتستخدم الثانية ٣×٣.

**لا تكتب بُعد التسطيح صراحةً.** فكل طالب يكتب `nn.Linear(1568, 64)` مرّة، ثم يغيّر حجم الصورة فيلقى

```
RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x400 and 800x10)
```

— وهو نشاط الصباح الأول. وللخروج طريقان صادقان: `nn.LazyLinear` تستنتج البُعد من أول دفعة تراها،
و`nn.AdaptiveAvgPool2d((7, 7))` تذهب أبعد فتُثبّت الشبكة التي يستقبلها الرأس مهما دخل. استخدم
التجميع التكيّفي: فهو ما يجعل فحص «يقبل ٣٢×٣٢ أيضًا» ينجح، وهو بعينه ما تفعله `resnet18` قبل مصنّفها
— ولهذا يعمل الضبط الدقيق يوم الخميس على صور ليست ٢٢٤×٢٢٤ كذلك.

</div>

In [ ]:
POOLED_GRID = 7


class SmallCNN(nn.Module):
    """Two conv-ReLU-pool blocks and a two-layer head, with no hard-coded flatten size."""
    # TODO: average pool to POOLED_GRID, and a flatten-linear-ReLU-linear head.
    # مهمة: متوسّطًا تكيّفيًا إلى `POOLED_GRID`، ورأسًا: تسطيح-خطّية-ReLU-خطّية.


torch.manual_seed(42)
model = SmallCNN()

print("28x28 batch ->", tuple(model(torch.zeros(2, 1, 28, 28)).shape))
print("32x32 batch ->", tuple(model(torch.zeros(2, 1, 32, 32)).shape), " (same model, no edits)")

### Task 2.2 — 432 against 393,216

`torchinfo.summary` prints the whole model with a parameter count per layer. Run it on your CNN,
then answer two questions from what it printed: how many parameters in total, and how many of them
sit in the **last** layers rather than the conv ones.

Then the arithmetic from the slide, on paper before code. The layer is `nn.Conv2d(3, 16, 3)`:
sixteen filters, each 3×3, each looking at 3 input channels.

```
16 × 3 × 3 × 3 = 432 weights, plus 16 biases
```

And a dense layer of 128 units on the same 32×32×3 image, flattened to 3,072:

```
3,072 × 128 = 393,216 weights
```

**432 against 393,216.** Nine hundred times fewer, and the conv layer is the one that can find the
same edge in the corner that it learned in the middle. Build both layers in PyTorch and let it
confirm your arithmetic rather than the other way round.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — ٤٣٢ مقابل ٣٩٣٬٢١٦

تطبع `torchinfo.summary` النموذج كاملًا مع عدد المعاملات لكل طبقة. شغّلها على شبكتك، ثم أجب عن
سؤالين مما طُبع: كم مجموع المعاملات، وكم منها يجلس في الطبقات **الأخيرة** لا في الالتفافية.

ثم حساب الشريحة، على الورق قبل الشيفرة. الطبقة `nn.Conv2d(3, 16, 3)`: ستة عشر مرشِّحًا، كلٌّ ٣×٣،
وكلٌّ ينظر إلى ثلاث قنوات دخل.

```
١٦ × ٣ × ٣ × ٣ = ٤٣٢ وزنًا، مع ١٦ انحيازًا
```

وطبقة كثيفة من ١٢٨ وحدة على الصورة نفسها ٣٢×٣٢×٣ مُسطَّحةً إلى ٣٬٠٧٢:

```
٣٬٠٧٢ × ١٢٨ = ٣٩٣٬٢١٦ وزنًا
```

**٤٣٢ مقابل ٣٩٣٬٢١٦.** أقلّ بتسعمئة ضعف تقريبًا، والطبقة الالتفافية هي التي تستطيع أن تجد في الزاوية
الحافة نفسها التي تعلّمتها في الوسط. ابنِ الطبقتين في PyTorch ودعه يؤكّد حسابك لا العكس.

</div>

In [ ]:
from torchinfo import summary

print(summary(model, input_size=(1, 1, 28, 28), verbose=0))

# TODO: Build the slide's conv layer and its dense rival, and count the weights in each.
# مهمة: ابنِ الطبقة الالتفافية من الشريحة ومنافستها الكثيفة، وعُدّ أوزان كلٍّ منهما.

print(f"\nnn.Conv2d(3, 16, 3):  16 x 3 x 3 x 3 = {CONV_WEIGHTS} weights + {CONV_BIASES} biases")
print(f"nn.Linear(3072, 128): 3072 x 128     = {DENSE_WEIGHTS:,} weights")
print(f"ratio: {DENSE_WEIGHTS / CONV_WEIGHTS:,.0f}x more parameters, on the same image")

conv_params = sum(p.numel() for p in model.features.parameters())
head_params = sum(p.numel() for p in model.head.parameters())
print(f"\nyour CNN: {conv_params:,} parameters in the conv blocks, "
      f"{head_params:,} in the head — the head is {head_params / (conv_params + head_params):.0%} of it")

### Task 2.3 — train it

Five epochs, `Adam` at 1e-3, batches of 128. Record four numbers per epoch: train loss, train
accuracy, validation loss, validation accuracy. One row per epoch, in a DataFrame — task 2.4 plots
it and it is half the artefact.

`model.train()` inside the epoch and `model.eval()` with `torch.no_grad()` to score it. That pair
does nothing visible in this model, which is exactly why it is worth building the habit now: on
Thursday there is dropout and batch-norm in the network and forgetting it silently changes your
numbers.

Expect roughly 8 seconds an epoch. The target is **≥ 0.98 on the test set**, which this
architecture passes on epoch 2 and finishes near 0.99.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — درّبها

خمس حقب، و`Adam` عند ١e−٣، ودفعات من ١٢٨. وسجّل أربعة أعداد لكل حقبة: خسارة التدريب ودقّته، وخسارة
التحقّق ودقّته. صف واحد لكل حقبة في `DataFrame` — فالمهمة ٢٫٤ ترسمه وهو نصف الأثر.

و`model.train()` داخل الحقبة و`model.eval()` مع `torch.no_grad()` للتقييم. وهذا الزوج لا يفعل شيئًا
مرئيًا في هذا النموذج، وهذا بالضبط سبب بناء العادة الآن: ففي الخميس يوجد Dropout وتوحيد دفعات في
الشبكة، ونسيانه يغيّر أرقامك بصمت.

وتوقّع نحو ثماني ثوانٍ للحقبة. والهدف **٠٫٩٨ فأعلى على مجموعة الاختبار**، وهذه البنية تتجاوزه في
الحقبة الثانية وتنتهي قرب ٠٫٩٩.

</div>

In [ ]:
import time

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_set, batch_size=512, shuffle=False, num_workers=0)


def evaluate(model, loader, loss_fn):
    """Mean loss and accuracy over a loader, with the model in eval mode."""
    # TODO: count, then return the two means.
    # مهمة: ثم أرجِع المتوسّطين.


def train(model, epochs=EPOCHS):
    """Train for `epochs` and return one row of metrics per epoch."""
    # TODO: zero_grad/backward/step, then evaluate() — one row of metrics per epoch.
    # مهمة: التدرّجات والتمرير الخلفي والخطوة، ثم `evaluate()` — وصف مقاييس لكل حقبة.


torch.manual_seed(42)
model = SmallCNN()
curves = train(model)
FINAL_ACCURACY = float(curves["val_accuracy"].iloc[-1])
print(f"\nfinal test accuracy: {FINAL_ACCURACY:.4f}")

### Task 2.4 — plot the curves, then say which shape you got

Two panels: loss against epoch, accuracy against epoch, train and validation on each. Then use
W3D5's vocabulary — the four curve shapes from Friday — and write which one this is in
`DIAGNOSIS`. The sanity check counts your words; the TA reads them.

The four shapes, as a reminder: **diverging** (loss climbs — learning rate too high),
**flat** (nothing moves — learning rate too low, or dead units), **overfitting** (train falls,
validation turns and climbs), and **healthy** (both fall, validation slightly above train, gap
stable).

<div dir="rtl" align="right">

### المهمة ٢٫٤ — ارسم المنحنيين ثم قل أي شكل حصل عندك

لوحتان: الخسارة مقابل الحقبة، والدقة مقابل الحقبة، وفي كلٍّ منهما التدريب والتحقّق. ثم استخدم مفردات
الأسبوع الثالث اليوم الخامس — الأشكال الأربعة من الجمعة — واكتب أيّها هذا في `DIAGNOSIS`. فحص السلامة
يعدّ كلماتك، والمساعد يقرؤها.

والأشكال الأربعة تذكيرًا: **متباعد** (الخسارة تصعد — معدّل التعلّم مرتفع جدًا)، و**مسطّح** (لا شيء
يتحرّك — معدّل التعلّم منخفض جدًا أو وحدات ميتة)، و**فرط مطابقة** (التدريب ينزل والتحقّق يستدير
ويصعد)، و**سليم** (كلاهما ينزل، والتحقّق أعلى قليلًا من التدريب، والفجوة مستقرّة).

</div>

In [ ]:

# TODO: Plot loss and accuracy against epoch, training and validation on each panel.
# مهمة: ارسم الخسارة والدقة مقابل الحقبة، والتدريب والتحقّق في كل لوحة.

DIAGNOSIS = (
    # TODO: Name the curve shape and say what in the plot tells you that, in one sentence.
    # مهمة: سمِّ شكل المنحنى وقل ما الذي في الرسم يدلّك على ذلك، في جملة واحدة.
)
print(DIAGNOSIS)
print(f"\ngap at the last epoch: train {curves['train_accuracy'].iloc[-1]:.4f} "
      f"vs validation {curves['val_accuracy'].iloc[-1]:.4f}")

### Task 2.5 — the confusion matrix, and five pictures

A 10×10 matrix: rows are the true digit, columns the predicted one. The diagonal is what it got
right; everything off it is a mistake with a name.

Find the **largest off-diagonal cell** — the single pair the model confuses most — and display five
test images from it. Look at them before you draw any conclusion. On this architecture the worst
pair is usually `4 → 9` with about eight cases out of ten thousand, and when you look at those
eights you will find handwriting that a person would also hesitate over.

That is the point of the task. A confusion matrix tells you *where* to look; only the images tell
you whether the model is wrong or the label is optimistic.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — مصفوفة الالتباس، وخمس صور

مصفوفة ١٠×١٠: الصفوف الرقم الحقيقي والأعمدة الرقم المتوقَّع. والقطر ما أصابه، وكل ما خارجه خطأ له اسم.

جِد **أكبر خانة خارج القطر** — أي الزوج الذي يخلط بينه النموذج أكثر — واعرض خمس صور اختبار منه. وانظر
إليها قبل أن تستنتج شيئًا. وعلى هذه البنية يكون أسوأ زوج عادةً `٤ ← ٩` بنحو ثماني حالات من عشرة
آلاف، وحين تنظر إلى تلك الثماني تجد خطًّا يتردّد فيه الإنسان أيضًا.

وهذا هو مقصد المهمة. فمصفوفة الالتباس تدلّك **أين** تنظر، والصور وحدها تقول لك أالنموذج مخطئ أم
التسمية متفائلة.

</div>

In [ ]:

# TODO: Predict the whole test set, then build the 10x10 confusion matrix from it.
# مهمة: تنبّأ بمجموعة الاختبار كاملة، ثم ابنِ منها مصفوفة الالتباس ١٠×١٠.

off_diagonal = confusion.copy()
np.fill_diagonal(off_diagonal, 0)
WORST_TRUE, WORST_PREDICTED = np.unravel_index(off_diagonal.argmax(), off_diagonal.shape)
WORST_COUNT = int(off_diagonal.max())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
image = axes[0].imshow(confusion, cmap="Blues")
axes[0].set_xlabel("predicted"); axes[0].set_ylabel("true")
axes[0].set_xticks(range(10)); axes[0].set_yticks(range(10))
axes[0].set_title(f"confusion matrix — worst pair {WORST_TRUE} -> {WORST_PREDICTED} "
                  f"({WORST_COUNT} cases)")
fig.colorbar(image, ax=axes[0], fraction=0.046)
axes[1].axis("off")
plt.tight_layout(); plt.show()

# TODO: Show five test images whose true label is WORST_TRUE but were called WORST_PREDICTED.
# مهمة: اعرض خمس صور اختبار تسميتها الحقيقية `WORST_TRUE` وسُمِّيت `WORST_PREDICTED`.

print(f"worst pair: a true {WORST_TRUE} called {WORST_PREDICTED}, {WORST_COUNT} times out of "
      f"{int((truths == WORST_TRUE).sum())} {WORST_TRUE}s")

### Task 2.6 — draw the filters

The first conv layer holds 16 filters of 5×5. Pull the weight tensor out of the trained model and
draw all sixteen as small greyscale images, on one grid.

Look at them. Several have a bright side and a dark side with a boundary between — a light-to-dark
gradient in some direction. **Those are edge detectors.** They are the same objects you typed by
hand yesterday, and nobody typed these. There was no term in the loss function for "find edges";
there was only "name the digit", and edges turned out to be the useful thing to notice first.

Write one sentence in `FILTER_NOTE` about what you see. Being honest that some of them look like
nothing recognisable is a better answer than pretending all sixteen are textbook edge detectors —
at 5×5 and five epochs, several are still noise.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — ارسم المرشِّحات

تحمل الطبقة الالتفافية الأولى ستة عشر مرشِّحًا بحجم ٥×٥. أخرِج مصفوفة الأوزان من النموذج المُدرَّب
وارسم الستة عشر جميعًا صورًا رمادية صغيرة في شبكة واحدة.

وانظر إليها. لعدّة منها جانب مضيء وجانب مظلم بينهما حدّ — أي تدرّج من النصوع إلى العتمة في اتّجاه ما.
**تلك كواشف حواف.** وهي الأشياء نفسها التي كتبتها بيدك أمس، ولم يكتب أحد هذه. لم يكن في دالة الخسارة
حدٌّ يقول «جِد الحواف»، بل كان فيها «سمِّ الرقم» فقط، فتبيّن أن الحواف هي الشيء المفيد أن يُلاحَظ أولًا.

واكتب جملة في `FILTER_NOTE` عمّا تراه. والصدق في أن بعضها لا يشبه شيئًا معروفًا جواب أفضل من ادّعاء
أن الستة عشر كلها كواشف حواف كتابية — فعند ٥×٥ وخمس حقب يبقى عدد منها ضجيجًا.

</div>

In [ ]:

# TODO: Take the first conv layer's weights and draw all of them on one grid, then save it.
# مهمة: خُذ أوزان الطبقة الالتفافية الأولى وارسمها كلها في شبكة واحدة ثم احفظها.

FILTER_NOTE = (
    # TODO: One sentence: what do these sixteen 5x5 grids look like, honestly?
    # مهمة: جملة واحدة: كيف تبدو هذه الشبكات الستة عشر ٥×٥ بصدق؟
)
print(f"{FILTER_COUNT} filters of {filters.shape[2]}x{filters.shape[3]}, saved to {FILTER_FIGURE.name}")
print(FILTER_NOTE)

## Section 3 — Stretch: the same architecture, harder data  (≈30 min)

Change the dataset. Change nothing else — same class, same seed, same epochs, same optimiser, same
learning rate. Fashion-MNIST is 70,000 images of clothing in the same 28×28 greyscale shape, so it
drops in without touching a line of the model.

The accuracy falls by about **nine points**, to roughly 0.896. Build the confusion matrix and read
the top few pairs: shirt against T-shirt, pullover against coat, pullover against shirt. Those are
genuinely hard at 28×28 — a person shown the same thumbnails does not do much better.

Then write two sentences on why an MNIST accuracy is a misleading number to quote. The dataset
changed, the model did not, and nine points evaporated.

<div dir="rtl" align="right">

## القسم الثالث — الإضافي: البنية نفسها وبيانات أصعب (نحو ٣٠ دقيقة)

غيّر مجموعة البيانات ولا تغيّر شيئًا آخر: الصنف نفسه، والبذرة نفسها، والحقب نفسها، والمُحسِّن نفسه،
ومعدّل التعلّم نفسه. فـFashion-MNIST سبعون ألف صورة ملابس بالشكل الرمادي نفسه ٢٨×٢٨، فتدخل مكانها بلا
تغيير سطر في النموذج.

وتهبط الدقة نحو **تسع نقاط** إلى قرابة ٠٫٨٩٦. ابنِ مصفوفة الالتباس واقرأ أعلى أزواجها: القميص مقابل
التي‑شيرت، والكنزة مقابل المعطف، والكنزة مقابل القميص. وهذه صعبة فعلًا عند ٢٨×٢٨ — فالإنسان الذي
يُعرَض عليه المصغَّرات نفسها لا يفعل أفضل بكثير.

ثم اكتب جملتين في سبب كون الدقة المقتبَسة من MNIST رقمًا مُضلِّلًا. تغيّرت البيانات ولم يتغيّر النموذج،
فتبخّرت تسع نقاط.

</div>

In [ ]:

# TODO: report the accuracy drop and the three worst confused pairs.
# مهمة: مقدار هبوط الدقة وأسوأ ثلاثة أزواج ملتبسة.

print(f"\nMNIST {FINAL_ACCURACY:.4f}  ->  Fashion-MNIST {FASHION_ACCURACY:.4f}   "
      f"({100 * (FINAL_ACCURACY - FASHION_ACCURACY):.1f} points lost, same model)")
for count, true_index, predicted_index in pairs:
    print(f"  {names[true_index]} called {names[predicted_index]}: {count} times")

WHY_MNIST_MISLEADS = (
    # TODO: Two sentences on why an MNIST number should not be quoted as evidence of skill.
    # مهمة: جملتان في سبب ألّا يُقتبس رقم MNIST دليلًا على المهارة.
)
print("\n" + WHY_MNIST_MISLEADS)

## Save your artefacts

Two files. `cnn_mnist.pt` is the trained state dict plus enough configuration to rebuild the model
that produced it; `curves.parquet` is one row per epoch. Thursday and Friday both load a state dict
this way, and week 8's deployment lab loads exactly this shape.

Note what is saved: the **state dict**, not the model object. Pickling a model pickles the class
definition's import path with it, and it breaks the moment the file moves.

<div dir="rtl" align="right">

## احفظ آثارك

ملفّان. `cnn_mnist.pt` هو قاموس الحالة المُدرَّب مع ما يكفي من الإعدادات لإعادة بناء النموذج الذي
أنتجه، و`curves.parquet` صف لكل حقبة. ويُحمّل يوما الخميس والجمعة قاموس حالة بهذه الطريقة، ويُحمّل
معمل النشر في الأسبوع الثامن هذا الشكل بعينه.

ولاحظ ما يُحفظ: **قاموس الحالة** لا كائن النموذج. فتخليل النموذج يُخلِّل معه مسار استيراد تعريف الصنف،
وينكسر لحظة انتقال الملف.

</div>

In [ ]:
MODEL_PATH = ARTEFACT_DIR / "cnn_mnist.pt"
CURVES_PATH = ARTEFACT_DIR / "curves.parquet"

torch.save({"state_dict": model.state_dict(),
            "architecture": "SmallCNN",
            "in_channels": 1,
            "n_classes": 10,
            "pooled_grid": POOLED_GRID,
            "epochs": EPOCHS,
            "seed": 42,
            "test_accuracy": FINAL_ACCURACY}, MODEL_PATH)
curves.to_parquet(CURVES_PATH, index=False)

print(f"wrote {MODEL_PATH.name} ({MODEL_PATH.stat().st_size / 1024:.0f} KB) and "
      f"{CURVES_PATH.name} ({len(curves)} rows)")
print(curves.round(4).to_string(index=False))

## Sanity check

The last check reloads the file you just wrote into a fresh model and re-scores it. A state dict
that does not reproduce its own accuracy is a state dict you cannot deploy, and finding that out
now costs a minute rather than a week.

<div dir="rtl" align="right">

## فحص سلامة

يُعيد الفحص الأخير تحميل الملف الذي كتبته للتوّ في نموذج جديد ويُعيد تقييمه. فقاموس الحالة الذي لا
يُعيد إنتاج دقّته لا يمكن نشره، واكتشاف ذلك الآن يكلّف دقيقة لا أسبوعًا.

</div>

In [ ]:
check(model(torch.zeros(2, 1, 32, 32)).shape == (2, 10),
      "the model must accept a 32x32 image as well as a 28x28 one — a hard-coded flatten "
      "dimension is the usual reason it cannot",
      "يجب أن يقبل النموذج صورة ٣٢×٣٢ كما يقبل ٢٨×٢٨ — والسبب المعتاد للعجز أن بُعد التسطيح "
      "مكتوب صراحةً")

check(CONV_WEIGHTS == 432 and DENSE_WEIGHTS == 393216,
      f"nn.Conv2d(3, 16, 3) holds 432 weights and nn.Linear(3072, 128) holds 393,216 — "
      f"got {CONV_WEIGHTS} and {DENSE_WEIGHTS:,}",
      f"تحمل `nn.Conv2d(3, 16, 3)` أربعمئة واثنين وثلاثين وزنًا وتحمل `nn.Linear(3072, 128)` "
      f"٣٩٣٬٢١٦ — والناتج {CONV_WEIGHTS} و{DENSE_WEIGHTS:,}")

check(FINAL_ACCURACY >= 0.98,
      f"test accuracy should reach 0.98 in {EPOCHS} epochs, got {FINAL_ACCURACY:.4f}",
      f"يجب أن تبلغ دقة الاختبار ٠٫٩٨ في {EPOCHS} حقب، والناتج {FINAL_ACCURACY:.4f}")

check(len(curves) == EPOCHS and curves["epoch"].tolist() == list(range(1, EPOCHS + 1)),
      f"curves.parquet needs exactly one row per epoch — got {len(curves)} rows",
      f"يحتاج `curves.parquet` صفًّا واحدًا لكل حقبة تمامًا — والموجود {len(curves)} صفًّا")

check(FILTER_FIGURE.exists() and FILTER_COUNT == 16,
      f"the filter grid must be saved and show all 16 first-layer filters — file exists: "
      f"{FILTER_FIGURE.exists()}, filters drawn: {FILTER_COUNT}",
      f"يجب حفظ شبكة المرشِّحات وعرض المرشِّحات الستة عشر كلها — الملف موجود: "
      f"{FILTER_FIGURE.exists()}، والمرسوم: {FILTER_COUNT}")

reloaded = SmallCNN(in_channels=1, n_classes=10)
reloaded.load_state_dict(torch.load(MODEL_PATH, weights_only=True)["state_dict"])
_, reloaded_accuracy = evaluate(reloaded, DataLoader(test_set, batch_size=512), nn.CrossEntropyLoss())
check(reloaded_accuracy == FINAL_ACCURACY,
      f"reloading cnn_mnist.pt must reproduce the accuracy exactly — saved {FINAL_ACCURACY:.4f}, "
      f"reloaded {reloaded_accuracy:.4f}",
      f"يجب أن تُعيد قراءة `cnn_mnist.pt` الدقة نفسها تمامًا — المحفوظ {FINAL_ACCURACY:.4f} "
      f"والمُعاد {reloaded_accuracy:.4f}")

check(len(DIAGNOSIS.split()) >= 12 and len(FILTER_NOTE.split()) >= 12,
      f"tasks 2.4 and 2.6 want sentences, not placeholders — got {len(DIAGNOSIS.split())} and "
      f"{len(FILTER_NOTE.split())} words",
      f"تريد المهمّتان ٢٫٤ و٢٫٦ جملًا لا عبارات نائبة — والموجود {len(DIAGNOSIS.split())} و"
      f"{len(FILTER_NOTE.split())} كلمة")

report()

## What's next

**W4D3 — Detection with YOLO.** Tomorrow the model stops answering "which of ten classes is this
picture" and starts answering "what is in this picture, and where" — a list whose length it decides
for itself. You will run a real detector over the same twenty photographs you convolved on Monday,
then score it properly: IoU, a threshold sweep, non-max suppression, and mAP against ground truth.

Bring today's habit of looking at the failures. Two of those twenty photographs are there because
the detector fails on them.

<div dir="rtl" align="right">

## ما التالي

**الأسبوع ٤ اليوم ٣ — الكشف بـYOLO.** غدًا يتوقّف النموذج عن الإجابة عن «أيّ الفئات العشر هذه الصورة»
ويبدأ الإجابة عن «ماذا في هذه الصورة وأين» — قائمةً يقرّر هو طولها. وستُشغّل كاشفًا حقيقيًا على الصور
العشرين نفسها التي التففتها يوم الاثنين، ثم تُقيّمه كما يجب: تداخل الصناديق، ومسح العتبات، وكبت غير
الأقصى، وmAP مقابل المرجع.

واحمل معك عادة اليوم في النظر إلى حالات الفشل. فصورتان من تلك العشرين موجودتان لأن الكاشف يفشل فيهما.

</div>